# Beginner Friendly Notebook (4 STEPS ONLY ✨)
This notebook gives you a simple, clean, and reproducible baseline for the AIMO competition, you can then build upon it as you go :) 

We will use a famous approach called **Zero-Shot CoT** introduced by ([Kojima et al. 2022](https://arxiv.org/abs/2205.11916)), and it follows a simple prompting strategy that adds the phrase "Let's think step by step" This helps an LLM break the problem down logically.:
```
Q: A juggler can juggle 16 balls. Half of the balls are golf balls, 
and half of the golf balls are blue. How many blue golf balls are 
there?
A:Let’s think step by step.

(Output) 
There are 16 balls in total. Half of the balls are golf 
balls. That means that there are 8 golf balls. Half of the golf balls 
are blue. That means that there are 4 blue golf balls.
```

# Contents
1. Load Model  
2. Build Prompt and Inference  
3. Predict Test
4. Run everything


So let's get started my friends :)

In [1]:
import subprocess

subprocess.run(["pip", "uninstall", "--yes", "tensorflow", "matplotlib", "keras", "scikit-learn"])

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0
Found existing installation: matplotlib 3.7.2
Uninstalling matplotlib-3.7.2:
  Successfully uninstalled matplotlib-3.7.2
Found existing installation: keras 3.8.0
Uninstalling keras-3.8.0:
  Successfully uninstalled keras-3.8.0
Found existing installation: scikit-learn 1.2.2
Uninstalling scikit-learn-1.2.2:
  Successfully uninstalled scikit-learn-1.2.2


CompletedProcess(args=['pip', 'uninstall', '--yes', 'tensorflow', 'matplotlib', 'keras', 'scikit-learn'], returncode=0)

In [2]:
# This is a hidden cell why are you looking at 🙂🌚?!

import os, warnings

# Avoid TensorFlow/Flax paths in transformers → fewer protobuf issues
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"


# Use Python protobuf implementation (dodges GetPrototype crash)
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"



warnings.filterwarnings("ignore")

# 1. Load LLM

In [3]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Runing on: {device}")

Runing on: cuda


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_llm_model(model_id, device="cuda"):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    
    model.eval()
    return model, tokenizer

In [5]:
m_id = "/kaggle/input/qwen-3/transformers/4b-fp8/1" 
math_model, math_tokenizer = load_llm_model(m_id, device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# 2. Build Prompt and Inference

In [6]:
def build_prompt(tokenizer, question, style_hint):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an elite International Mathematical Olympiad problem solver.You are solving a national/international-level mathematics olympiad problem. You must rigorously define all variables, explore multiple solution strategies before committing, perform full case analysis where required, justify every nontrivial step, explicitly check boundary cases and hidden assumptions, and verify the final result using at least one independent method. Return only the final numerical answer inside \\boxed{}. The answer must be an integer in [0, 99999]. Never guess.Solve the problem with full rigor. After obtaining a candidate solution, actively attempt to refute your own answer by searching for counterexamples, re-running the logic from a different viewpoint, and stress-testing edge cases. Only after the answer survives refutation, return it in \\boxed{}. The answer must be an integer in [0, 99999]. Never guess.Solve this problem as if under IMO-level time pressure: identify the key invariant, symmetry, or extremal principle early, avoid brute force unless strictly justified, compress reasoning without sacrificing correctness, and perform at least one final arithmetic verification pass. Return only the final integer answer in \\boxed{}, with 0 ≤ answer ≤ 99999. Never guess.You must attempt at least two fundamentally different solution approaches (e.g., algebraic vs geometric, combinatorial vs number-theoretic). Proceed with the more rigorous one and use the other as a verification tool. Return only the verified final answer in \\boxed{}, where the answer is an integer in [0, 99999]. Never guess.Solve the problem rigorously. If at any point a step relies on an unproven assumption, a jump in logic is detected, or the computation becomes inconsistent, you must restart the solution from first principles. Return only the final verified integer answer inside \\boxed{}, with 0 ≤ answer ≤ 99999. Never guess.\n\n"
                "Rules:\n"
                "- Show your work by reasoning step-by-step.\n" # Change: Allow reasoning
                "- Perform consistency and sanity checks.\n"
                "- The final answer must be a non-negative integer between 0 and 99999.\n"
                "- End your response with 'The final answer is \\boxed{N}'." # Change: Specific trigger
            ),
        },
        {
            "role": "user",
            "content": (
                f"Problem:\n{question}\n\n"
                f"Guidance:\n{style_hint}\n\n"
                "Solve the problem and give the final answer."
            ),
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


In [11]:
def inference(model, tokenizer, prompt, device="cuda", max_new_tokens=3072):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # CRITICAL: deterministic
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


import re

def extract_final_answer(output):
    matches = re.findall(r"\\boxed\{([^}]+)\}", output)
    if not matches:
        return None

    candidate = matches[-1].strip()
    nums = re.findall(r"\d+", candidate)
    if not nums:
        return None

    val = int(nums[-1])
    if 0 <= val <= 99999:
        return val
    return None


# 3. Predict Test

In [12]:
STYLE_1 = "Solve systematically using algebraic or explicit derivations."
STYLE_2 = "Look for symmetry, invariants, or clever Olympiad-style insights."

def solve_problem(model, tokenizer, question, device="cuda"):
    # Pass 1
    prompt1 = build_prompt(tokenizer, question, STYLE_1)
    out1 = inference(model, tokenizer, prompt1, device)
    ans1 = extract_final_answer(out1)

    # Pass 2
    prompt2 = build_prompt(tokenizer, question, STYLE_2)
    out2 = inference(model, tokenizer, prompt2, device)
    ans2 = extract_final_answer(out2)

    # Decision logic
    if ans1 is not None and ans1 == ans2:
        return ans1

    # If mismatch, choose safer one
    candidates = [a for a in [ans1, ans2] if a is not None]
    if candidates:
        return candidates[0]

    return 0  # safe fallback


In [13]:
model_id = "/kaggle/input/qwen-3/transformers/4b-fp8/1"
model, tokenizer = load_llm_model(model_id)

def predict_df(problem_text):
    return solve_problem(model, tokenizer, problem_text)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [14]:
# def predict_df(model, tokenizer, df, id_col="id", question_col="problem"):
#     preds = []
#     for _, row in df.iterrows():
#         q = row[question_col]
#         prompt = build_prompt_reasoning(tokenizer, q)
#         raw = inference(model, tokenizer, prompt, device=device)
#         ans = extract_final_answer(raw)
        
#         if ans <= -1: 
#             ans = 0
                
#         preds.append({id_col: row[id_col], "answer": ans})
#     return pd.DataFrame(preds)

## Local Inference Code for showing the reasoning and the final output to check how better

In [ ]:
import pandas as pd

def run_local_evaluation(model, tokenizer, reference_path):
    # 1. Load the reference data
    df = pd.read_csv(reference_path)
    
    results = []
    
    print(f"--- Starting Evaluation on {len(df)} problems ---\n")
    
    for i, row in df.iterrows():
        question = row['problem']
        expected_answer = row['answer']
        
        print(f"PROMPT {i+1}: {question[:100]}...")
        
        # Pass 1: Systematic
        prompt1 = build_prompt(tokenizer, question, STYLE_1)
        raw_output1 = inference(model, tokenizer, prompt1)
        ans1 = extract_final_answer(raw_output1)
        
        # Pass 2: Olympiad Insight
        prompt2 = build_prompt(tokenizer, question, STYLE_2)
        raw_output2 = inference(model, tokenizer, prompt2)
        ans2 = extract_final_answer(raw_output2)
        
        # Print Reasoning for Inspection
        print(f"\n[PASS 1 REASONING]:\n{raw_output1}")
        print(f"-"*30)
        print(f"[PASS 2 REASONING]:\n{raw_output2}")
        
        # Check correctness
        final_ans = ans1 if (ans1 == ans2 and ans1 is not None) else (ans1 if ans1 is not None else 0)
        is_correct = (final_ans == expected_answer)
        
        print(f"\nFinal Prediction: {final_ans} | Expected: {expected_answer}")
        print(f"Status: {'✅ CORRECT' if is_correct else '❌ WRONG'}")
        print("="*50)
        
        results.append({
            "id": i,
            "correct": is_correct,
            "ans1": ans1,
            "ans2": ans2,
            "expected": expected_answer
        })

    # Summary
    score = sum([1 for r in results if r['correct']])
    print(f"\nTotal Score: {score}/{len(df)}")
reference_path = "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
run_local_evaluation(math_model, math_tokenizer, reference_path)

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--- Starting Evaluation on 10 problems ---

PROMPT 1: Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie ...


# 
4. Run Everything 🫡
🎉 **Congratulations!**  
You made it all the way to a full working AIMO baseline.  
Give yourself a pat on the back 👏

Wasn't it simpler than you expected? 😊

Now you're ready to:
- explore better prompts  
- try self-consistency  
- experiment with bigger models  
- or even publish your own improvements  

In [ ]:
# import os
# import polars as pl
# import pandas as pd
# from transformers import AutoTokenizer, AutoModelForCausalLM
# import kaggle_evaluation.aimo_3_inference_server
# import torch
# import re

# class AIMOModel:
#     def __init__(self):
#         self.model = None
#         self.tokenizer = None
#         self.device = "cuda" if torch.cuda.is_available() else "cpu"

#     def load(self):
#         if self.model is None:
#             prt("Loading Qwen model...")
#             model_id = "/kaggle/input/qwen2.5-math/transformers/1.5b-instruct/1"
#             self.model, self.tokenizer = load_llm_model(
#                 model_id, device=self.device
#             )
#         return self.model, self.tokenizer

#     def predict_one(self, problem: str) -> int:
#         model, tokenizer = self.load()in

#         # ----- Pass 1 -----
#         prompt1 = build_prompt(tokenizer, problem, STYLE_1)
#         out1 = inference(model, tokenizer, prompt1, device=self.device)
#         ans1 = extract_final_answer(out1)

#         # ----- Pass 2 -----
#         prompt2 = build_prompt(tokenizer, problem, STYLE_2)
#         out2 = inference(model, tokenizer, prompt2, device=self.device)
#         ans2 = extract_final_answer(out2)

#         # ----- Decision Logic -----
#         if ans1 is not None and ans1 == ans2:
#             return ans1

#         if ans1 is not None:
#             return ans1

#         if ans2 is not None:
#             return ans2

#         return 0  # Safe fallback (never invalid)

# global_model = AIMOModel()

# def predict(id_: pl.Series, problem: pl.Series) -> pl.DataFrame:
#     problem_text = problem.item(0)
#     id_val = id_.item(0)

#     answer = global_model.predict_one(problem_text)
#     return pl.DataFrame({"id": [id_val], "answer": [answer]})


# pd.read_csv(
#     "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv"
# ).drop("answer", axis=1).to_csv("reference.csv", index=False)



In [ ]:

# inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

# if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
#     inference_server.serve()
# else:
#     inference_server.run_local_gateway(("reference.csv",))


In [ ]:
# sub_path = "submission.parquet"
# sub = pl.read_parquet(sub_path)


# print("Submission head:")
# print(sub.head())
# print("\nSubmission shape:", sub.shape)
# print("Submission dtypes:", sub.dtypes)